### 1. Jev 主要支持以下三种模式（Primitive 问答类型）
- Choice（单选分类模式）：从一组候选字典中选择一项，并返回每项的概率分布与置信度
- Noul（是非二分模式）：对一个陈述或问题判断是/否，返回介于 0 到 1 之间“为真”的概率。
 - Score（评分模式）：根据提供的从低到高的等级文本列表，对文本进行打分。

### 2. Jev (TypeSafe SDK) 三种核心模式 API 用法示例

In [1]:
from dotenv import load_dotenv

load_dotenv()
from typesafe_sdk import TypeSafeClient, Choice, Noul, Score

with TypeSafeClient() as client:
    # 模拟一段用户客服反馈文本作为输入状态
    customer_message = (
        "我这个月的 Pro 会员被重复扣款了两次！"
        "请立刻帮我把多扣的钱退掉，这已经是本周第二次出现系统故障了，非常愤怒！"
    )

    response = client.system_one(
        state={"message": customer_message},
        questions={
            # 1. Choice 模式：分类选择题（多选一，附带各选项概率及置信度）
            "department": Choice(
                instructions="应该将此消息分派给哪个团队处理？",
                criteria={
                    "billing": "涉及扣款、退款、账单或支付异常的问题",
                    "technical": "涉及应用崩溃、功能失效或系统故障的Bug",
                    "general": "其他普通咨询或账号建议"
                }
            ),

            # 2. Noul 模式：是非判断题（返回 0~1 的布尔成立概率）
            "is_angry": Noul(
                instructions="客户在文本中表现出了明显的愤怒或情绪失控？"
            ),

            # 3. Score 模式：量表评分题（根据指定的等级从低到高计算加权得分）
            "urgency": Score(
                instructions="评估该工单的紧急程度",
                criteria=[
                    "不紧急：常规询问，不影响日常使用",
                    "一般：功能受损但有临时替代方案",
                    "紧急：核心资产受损（如重复扣款/服务完全中断），需立刻人工介入"
                ]
            ),
        },
    )

### 3. 读取和解析返回结果

#### 3.1 读取 Choice 结果

In [2]:
dept_answer = response.choices["department"]
print(f"【Choice 分类模式】")
print(f"  - 最佳选择: {dept_answer.choice}")
print(f"  - 置信度 (Confidence): {dept_answer.confidence}")
print(f"  - 完整概率分布: {dept_answer.probabilities}\n")

【Choice 分类模式】
  - 最佳选择: billing
  - 置信度 (Confidence): 1.0
  - 完整概率分布: {'billing': 1.0, 'technical': 0.0, 'general': 0.0}



#### 3.2 读取 Noul 结果

In [3]:
angry_answer = response.nouls["is_angry"]
print(f"【Noul 是非模式】")
print(f"  - 为真的概率 (Probability): {angry_answer.noul}")
# 通常通过设定阈值转为布尔动作
is_customer_angry = angry_answer.noul > 0.8
print(f"  - 阈值判定 (>0.8): {'触发情绪安抚流程' if is_customer_angry else '正常处理'}\n")

【Noul 是非模式】
  - 为真的概率 (Probability): 0.95
  - 阈值判定 (>0.8): 触发情绪安抚流程



#### 3.3 读取 Score 结果

In [4]:
urgency_answer = response.scores["urgency"]
print(f"【Score 评分模式】")
print(f"  - 计算出的加权得分: {urgency_answer.score}")
print(f"  - 各等级的概率细分: {urgency_answer.probabilities}")

【Score 评分模式】
  - 计算出的加权得分: 2.0
  - 各等级的概率细分: {0: 0.0, 1: 0.0, 2: 1.0}
